In [8]:
import requests
import pandas as pd
import json
import time
from datetime import datetime


In [9]:
import requests
import pandas as pd
import json
import time
from datetime import datetime

# Cấu hình chung giả lập trình duyệt
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# ==========================================
# 1. HÀM LẤY BÁO CÁO TÀI CHÍNH (ĐÃ SỬA LỖI)
# ==========================================
def get_finance_report(symbol, report_type='QUY'):
    """
    Hàm lấy dữ liệu Báo cáo tài chính (Kết quả kinh doanh).
    report_type: 'QUY' hoặc 'NAM'
    """
    print(f"📊 Đang tải BCTC ({report_type}) của {symbol}...")
    
    url = "https://cafef.vn/du-lieu/Ajax/PageNew/FinanceReport.ashx"
    
    # --- LOGIC FIX LỖI ENDDATE ---
    # Nếu là QUÝ: Phải truyền dạng "Tháng-Năm" (ví dụ 12-2030)
    # Nếu là NĂM: Truyền "Năm" (ví dụ 2030) là đủ
    if report_type == 'QUY':
        end_date = "12-2030" # Lấy tương lai xa để bao trọn dữ liệu mới nhất
    else:
        end_date = "2030"
        
    params = {
        "Symbol": symbol,
        "Type": 1,            # 1: KQKD, 2: CĐKT, 3: Lưu chuyển tiền tệ
        "ReportType": report_type,
        "TotalRow": 100,      # Lấy 100 kỳ gần nhất
        "EndDate": end_date   # <--- THAM SỐ QUAN TRỌNG ĐÃ FIX
    }
    
    try:
        r = requests.get(url, params=params, headers=HEADERS)
        
        # Kiểm tra nếu server trả về lỗi HTTP
        if r.status_code != 200:
            print(f"❌ Server Error: {r.status_code}")
            return pd.DataFrame()
            
        data_json = r.json()
        
        # Kiểm tra xem có dữ liệu bên trong không
        if 'Data' not in data_json or data_json['Data'] is None:
            print(f"⚠️ Không có dữ liệu cho loại báo cáo {report_type}")
            return pd.DataFrame()
            
        data_values = data_json['Data']['Value']
        
        # Xử lý làm phẳng JSON (Flatten)
        processed_data = []
        for period in data_values:
            row = {
                'Nam': period['Year'],
                'Ky': period.get('Quater', 0), # 0 nếu là báo cáo Năm
                'LoaiBC': report_type
            }
            
            # Lặp qua các chỉ số (Doanh thu, Lợi nhuận...)
            for item in period['Value']:
                row[item['Code']] = item['Value']
            
            processed_data.append(row)
            
        df = pd.DataFrame(processed_data)
        
        # Sắp xếp lại theo Năm/Kỳ giảm dần để dễ nhìn
        if not df.empty:
            df = df.sort_values(by=['Nam', 'Ky'], ascending=False).reset_index(drop=True)
            
        return df
        
    except Exception as e:
        print(f"❌ Lỗi ngoại lệ khi lấy BCTC {report_type}: {e}")
        return pd.DataFrame()

# ==========================================
# 2. HÀM LẤY LỊCH SỬ GIÁ (GIỮ NGUYÊN)
# ==========================================
def get_price_history(symbol):
    print(f"💰 Đang tải lịch sử giá của {symbol}...")
    url = "https://cafef.vn/du-lieu/Ajax/PageNew/DataHistory/PriceHistory.ashx"
    
    params = {
        "Symbol": symbol,
        "PageIndex": 1,
        "PageSize": 10000 # Lấy trọn gói 1 lần
    }
    
    try:
        r = requests.get(url, params=params, headers=HEADERS)
        raw_data = r.json()['Data']['Data']
        df = pd.DataFrame(raw_data)
        
        # Chuyển đổi định dạng ngày và số
        df['Ngay'] = pd.to_datetime(df['Ngay'], format='%d/%m/%Y')
        
        # Hàm clean số: xóa dấu phẩy
        def clean_num(x):
            if isinstance(x, str):
                return float(x.replace(',', ''))
            return x
            
        cols_to_clean = ['GiaDieuChinh', 'GiaDongCua', 'KhoiLuongKhopLenh', 'GiaTriKhopLenh']
        for col in cols_to_clean:
            if col in df.columns:
                df[col] = df[col].apply(clean_num)
                
        df = df.sort_values('Ngay').reset_index(drop=True)
        return df[['Ngay', 'GiaDieuChinh', 'GiaDongCua', 'KhoiLuongKhopLenh', 'GiaTriKhopLenh']]
    except Exception as e:
        print(f"Lỗi lấy giá: {e}")
        return pd.DataFrame()

# ==========================================
# 3. HÀM LẤY CHỈ SỐ TÀI CHÍNH (GIỮ NGUYÊN)
# ==========================================
def get_financial_ratios(symbol):
    print(f"📈 Đang tải Chỉ số tài chính của {symbol}...")
    url = "https://cafef.vn/du-lieu/Ajax/PageNew/GetDataChiSoTaiChinh.ashx"
    
    params = {
        "Symbol": symbol,
        "TotalRow": 100,
        "EndDate": "2030" # Năm tương lai dùng ok cho API này
    }
    
    try:
        r = requests.get(url, params=params, headers=HEADERS)
        data = r.json()['Data']['Value']
        
        processed_data = []
        for period in data:
            row = {'Nam': period['Year'], 'Ky': period.get('Quater', 0)}
            for item in period['Value']:
                row[item['Code']] = item['Value']
            processed_data.append(row)
            
        return pd.DataFrame(processed_data)
    except Exception as e:
        print(f"Lỗi lấy chỉ số: {e}")
        return pd.DataFrame()


In [10]:

# ==========================================
# CHẠY CHƯƠNG TRÌNH CHÍNH
# ==========================================
if __name__ == "__main__":
    ticker = "HRC" # Mã cổ phiếu cần lấy
    
    # 1. Lấy Báo Cáo Tài Chính (QUÝ & NĂM)
    df_fin_quarter = get_finance_report(ticker, 'QUY')
    df_fin_year = get_finance_report(ticker, 'NAM')
    
    # 2. Lấy Lịch Sử Giá
    df_price = get_price_history(ticker)
    
    # 3. Lấy Chỉ Số Tài Chính
    df_ratios = get_financial_ratios(ticker)
    
    # --- HIỂN THỊ KẾT QUẢ ---
    print("\n" + "="*40)
    print(f"✅ HOÀN TẤT CRAWL DỮ LIỆU MÃ: {ticker}")
    print("="*40)
    
    if not df_fin_quarter.empty:
        print(f"\n1. BCTC QUÝ ({len(df_fin_quarter)} dòng):")
        print(df_fin_quarter[['Nam', 'Ky', 'DTTBHCCDV', 'LNSTTNDN']].head(3))
        # DTTBHCCDV: Doanh thu thuần, LNSTTNDN: Lợi nhuận sau thuế
    else:
        print("\n1. BCTC QUÝ: Không có dữ liệu!")

    if not df_fin_year.empty:
        print(f"\n2. BCTC NĂM ({len(df_fin_year)} dòng):")
        print(df_fin_year[['Nam', 'DTTBHCCDV', 'LNSTTNDN']].head(3))
        
    if not df_price.empty:
        print(f"\n3. LỊCH SỬ GIÁ ({len(df_price)} dòng):")
        print(df_price.tail(3))
        
    if not df_ratios.empty:
        print(f"\n4. CHỈ SỐ TÀI CHÍNH ({len(df_ratios)} dòng):")
        print(df_ratios[['Nam', 'Ky', 'EPS', 'PE', 'ROE']].head(3))

📊 Đang tải BCTC (QUY) của HRC...
📊 Đang tải BCTC (NAM) của HRC...
💰 Đang tải lịch sử giá của HRC...
📈 Đang tải Chỉ số tài chính của HRC...

✅ HOÀN TẤT CRAWL DỮ LIỆU MÃ: HRC

1. BCTC QUÝ (86 dòng):
    Nam  Ky   DTTBHCCDV    LNSTTNDN
0  2025   3  67228885.0  22043884.0
1  2025   2  59460549.0   3322067.0
2  2025   1  37348363.0   1427632.0

2. BCTC NĂM (19 dòng):
    Nam     DTTBHCCDV      LNSTTNDN
0  2024  2.143590e+08  6.140112e+07
1  2023  1.831741e+08  1.697786e+07
2  2022  1.792032e+08  1.015885e+07

3. LỊCH SỬ GIÁ (4751 dòng):
           Ngay  GiaDieuChinh  GiaDongCua  KhoiLuongKhopLenh  GiaTriKhopLenh
4748 2026-01-14         26.55       26.55                200         5310000
4749 2026-01-15         26.60       26.60               6600       176015000
4750 2026-01-16         28.45       28.45               4400       125155000

4. CHỈ SỐ TÀI CHÍNH (17 dòng):
    Nam  Ky   EPS     PE    ROE
0  2024   0  2.03   20.2  10.18
1  2021   0  0.73   72.6   4.01
2  2020   0  0.30  170.0  